# SAM 3 batch pixel annotation from the processed CMZ dataset

This notebook uses the friend's existing `cmz_processed_dataset.zip`: `metadata/manifest.csv` supplies video, behavior, lighting and split metadata, while `processed/full640` supplies the grayscale/denoised/CLAHE/letterboxed images. It does **not** use `crop224`, because segmentation must learn to locate the civet in the full scene.

SAM 3 outputs are annotation proposals, not automatic ground truth. Every proposal starts as `needs_review` and must be visually accepted, corrected in CVAT, or rejected before training.

## 1. Runtime, storage and SAM 3 installation

Colab: use a GPU runtime and add a Colab secret named `HF_TOKEN`. Local Windows: launch Jupyter from a CUDA-enabled Python environment, set `CMZ_PROJECT_ROOT`, and authenticate using `HF_TOKEN`, a cached Hugging Face login, or the secure interactive login prompt. The token is never written into this notebook.

In [1]:
import sys, subprocess
from pathlib import Path
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.2.2+cu118
GPU: NVIDIA GeForce RTX 2050
GPU memory (GB): 4.0


In [2]:
SAM3_DIR = Path('/content/sam3')
SAM3_DIR.parent.mkdir(parents=True, exist_ok=True)
if not SAM3_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/sam3.git', str(SAM3_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{SAM3_DIR}[notebooks]'], check=True)
print('SAM 3:', SAM3_DIR)
print('If imports fail after the first installation, restart the kernel once and continue below.')

Environment: local
SAM 3: C:\Users\ASUS TUF GAMING\.cache\cmz-sam3\sam3
If imports fail after the first installation, restart the kernel once and continue below.


If imports fail after the first installation, restart the Colab session or local Jupyter kernel once. For local Windows usage, set the project path before starting Jupyter, for example: `$env:CMZ_PROJECT_ROOT='G:\My Drive\CMZprojectphase2'; jupyter notebook`. You may optionally set `$env:HF_TOKEN='hf_...'` for that PowerShell session; otherwise the notebook opens a secure login prompt.

In [ ]:
import sys, subprocess, zipfile, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

drive.mount('/content/drive')
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN was not found in Colab Secrets.')
login(token=hf_token)
PROJECT_ROOT = Path('/content/drive/MyDrive/CMZprojectphase2')
SAM3_DIR = Path('/content/sam3')
LOCAL_EXTRACT_ROOT = Path('/content/CMZ')
ZIP_PATH = PROJECT_ROOT / 'data' / 'cmz_processed_dataset.zip'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'sam3_batch_annotation'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Processed dataset ZIP not found: {ZIP_PATH}')
print('Project root:', PROJECT_ROOT)
print('SAM 3 directory:', SAM3_DIR)
print('Dataset ZIP:', ZIP_PATH)
print('Annotation output:', OUTPUT_ROOT)

## 2. Extract and inspect the friend's processed dataset

Extraction uses Colab local storage for speed. Outputs and review files are saved to Drive.

In [ ]:
manifest_candidates = list(LOCAL_EXTRACT_ROOT.rglob('metadata/manifest.csv')) if LOCAL_EXTRACT_ROOT.exists() else []
if not manifest_candidates:
    LOCAL_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print('Extracting processed dataset to Colab local storage...')
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(LOCAL_EXTRACT_ROOT)
    manifest_candidates = list(LOCAL_EXTRACT_ROOT.rglob('metadata/manifest.csv'))

if len(manifest_candidates) != 1:
    raise RuntimeError(f'Expected one manifest.csv, found: {manifest_candidates}')
MANIFEST_PATH = manifest_candidates[0]
DATASET_BASE = MANIFEST_PATH.parent.parent
manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)

required = {'video_id', 'label', 'lighting', 'timestamp_sec', 'full_path'}
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f'Manifest is missing columns: {sorted(missing)}')

print('Dataset base:', DATASET_BASE)
print('Manifest rows:', len(manifest))
print('Videos:', manifest.video_id.nunique())
display(pd.crosstab(manifest['label'], manifest['lighting'], margins=True))

In [ ]:
sample_row = manifest[manifest['full_path'].notna()].iloc[0]
sample_path = DATASET_BASE / str(sample_row['full_path'])
print('Sample:', sample_path, 'exists:', sample_path.exists())
sample_image = Image.open(sample_path).convert('RGB')
print('Size:', sample_image.size)
display(sample_image)

## 3. Build a reproducible, video-diverse annotation selection

Frames are thinned to one candidate every two seconds, capped per video, and sampled within each available behavior-lighting group. Detector misses are included as hard-review candidates because a missing MegaDetector box does not prove that no civet is present.

The first pilot targets roughly 300-500 frames. Missing combinations such as night Eating are reported rather than fabricated.

In [ ]:
RANDOM_SEED = 42
SECONDS_BETWEEN_CANDIDATES = 2
TARGET_PER_LABEL_LIGHTING = 70
MAX_PER_VIDEO_PER_GROUP = 25
TARGET_DETECTOR_MISSES = 50
SELECTION_PATH = OUTPUT_ROOT / 'annotation_selection.csv'

def bool_series(frame, name, default=True):
    if name not in frame.columns:
        return pd.Series(default, index=frame.index)
    s = frame[name]
    if s.dtype == bool:
        return s.fillna(False)
    return s.astype(str).str.lower().isin(['true', '1', 'yes'])

if SELECTION_PATH.exists():
    selection = pd.read_csv(SELECTION_PATH)
    print('Reusing saved selection:', SELECTION_PATH)
else:
    candidates = manifest.copy()
    candidates = candidates[bool_series(candidates, 'readable', True)]
    candidates = candidates[candidates['label'].isin(['Eating', 'Locomotion', 'Resting'])]
    candidates = candidates[candidates['full_path'].notna() & candidates['video_id'].notna()]
    candidates['_time_bin'] = (pd.to_numeric(candidates['timestamp_sec'], errors='coerce') // SECONDS_BETWEEN_CANDIDATES).astype('Int64')
    candidates = candidates.dropna(subset=['_time_bin']).sort_values(['video_id', 'timestamp_sec'])
    candidates = candidates.drop_duplicates(['video_id', '_time_bin'])

    if 'box_x1' in candidates.columns:
        has_box = candidates['box_x1'].notna()
    elif 'civet_detected' in candidates.columns:
        has_box = bool_series(candidates, 'civet_detected', False)
    else:
        has_box = pd.Series(True, index=candidates.index)
    candidates['_has_prior_box'] = has_box

    chosen = []
    positives = candidates[candidates['_has_prior_box']].copy()
    for (label, lighting), group in positives.groupby(['label', 'lighting'], dropna=False):
        group = group.sample(frac=1, random_state=RANDOM_SEED)
        group = group.groupby('video_id', group_keys=False).head(MAX_PER_VIDEO_PER_GROUP)
        group = group.sample(min(TARGET_PER_LABEL_LIGHTING, len(group)), random_state=RANDOM_SEED)
        chosen.append(group.assign(annotation_kind='prior_detection'))

    misses = candidates[~candidates['_has_prior_box']].copy()
    if len(misses):
        misses = misses.sample(frac=1, random_state=RANDOM_SEED)
        misses = misses.groupby('video_id', group_keys=False).head(10)
        misses = misses.sample(min(TARGET_DETECTOR_MISSES, len(misses)), random_state=RANDOM_SEED)
        chosen.append(misses.assign(annotation_kind='detector_miss_review'))

    selection = pd.concat(chosen, ignore_index=True)
    selection.insert(0, 'selection_id', [f'A{i:05d}' for i in range(len(selection))])
    keep = ['selection_id', 'video_id', 'label', 'lighting', 'timestamp_sec', 'full_path', 'annotation_kind']
    for optional in ['split', 'fold', 'camera', 'window_id']:
        if optional in selection.columns:
            keep.append(optional)
    selection = selection[keep].sort_values(['label', 'lighting', 'video_id', 'timestamp_sec']).reset_index(drop=True)
    selection.to_csv(SELECTION_PATH, index=False)
    print('Saved selection:', SELECTION_PATH)

print('Selected frames:', len(selection))
display(pd.crosstab([selection['label'], selection['lighting']], selection['annotation_kind'], margins=True))
print('Frames per video (largest first):')
display(selection.video_id.value_counts().head(15).rename('frames').to_frame())

Inspect a random grid before running SAM 3. If the selection is dominated by near-duplicates or labels are visibly wrong, stop and correct the selection/manifest first.

In [ ]:
preview = selection.sample(min(24, len(selection)), random_state=RANDOM_SEED)
thumb_w, thumb_h, label_h, columns = 256, 256, 36, 6
rows = int(np.ceil(len(preview) / columns))
sheet = Image.new('RGB', (thumb_w*columns, (thumb_h+label_h)*rows), 'white')
draw = ImageDraw.Draw(sheet)
for i, row in enumerate(preview.itertuples()):
    path = DATASET_BASE / row.full_path
    image = Image.open(path).convert('RGB').resize((thumb_w, thumb_h))
    x, y = (i % columns)*thumb_w, (i // columns)*(thumb_h+label_h)
    sheet.paste(image, (x, y))
    draw.text((x+3, y+thumb_h+2), f'{row.selection_id} {row.label} {row.lighting}', fill='black')
display(sheet)

## 4. Load SAM 3

Paths are explicit to avoid tokenizer discovery errors. FP16 autocast is used for Tesla T4 inference; do not enable a global BF16 context.

In [ ]:
BPE_PATH = SAM3_DIR / 'sam3' / 'assets' / 'bpe_simple_vocab_16e6.txt.gz'
if not BPE_PATH.exists():
    raise FileNotFoundError(BPE_PATH)
CHECKPOINT_PATH = Path(hf_hub_download(repo_id='facebook/sam3', filename='sam3.pt', token=hf_token))
if str(SAM3_DIR) not in sys.path:
    sys.path.insert(0, str(SAM3_DIR))
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

gc.collect(); torch.cuda.empty_cache()
sam_model = build_sam3_image_model(
    bpe_path=str(BPE_PATH), checkpoint_path=str(CHECKPOINT_PATH),
    load_from_HF=False, device='cuda', eval_mode=True,
)
sam_processor = Sam3Processor(sam_model, device='cuda', confidence_threshold=0.5)
print('SAM 3 ready; dtype:', next(sam_model.parameters()).dtype)

## 5. Run resumable batch annotation

Each run processes the next pending batch. Increase `BATCH_LIMIT` only after confirming the first overlays look correct. Existing JSON records are skipped, so runtime disconnections do not erase completed work.

All proposals start as `needs_review`. A detector miss that SAM also misses is called `no_sam_detection`, not an accepted negative, until a person checks the source image.

In [ ]:
PROMPT = 'animal'
CONFIDENCE_THRESHOLD = 0.5
BATCH_LIMIT = 50
PROPOSAL_DIR = OUTPUT_ROOT / 'proposals'
PROPOSAL_DIR.mkdir(parents=True, exist_ok=True)

def to_numpy(value):
    return value.detach().float().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)

def make_overlay(image, mask, color=(0, 255, 0), alpha=0.45):
    result = np.asarray(image).copy()
    result[mask] = ((1-alpha)*result[mask] + alpha*np.asarray(color)).astype(np.uint8)
    return Image.fromarray(result)

def run_sam(image):
    sam_processor.set_confidence_threshold(CONFIDENCE_THRESHOLD)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        state = sam_processor.set_image(image)
        return sam_processor.set_text_prompt(state=state, prompt=PROMPT)

def record_path(selection_id):
    return PROPOSAL_DIR / selection_id / 'metadata.json'

pending = selection[~selection.selection_id.map(lambda x: record_path(x).exists())].head(BATCH_LIMIT)
print(f'Pending total: {sum(~selection.selection_id.map(lambda x: record_path(x).exists()))}; processing now: {len(pending)}')

In [ ]:
from tqdm.auto import tqdm

for row in tqdm(list(pending.itertuples()), desc='SAM 3 proposals'):
    item_dir = PROPOSAL_DIR / row.selection_id
    item_dir.mkdir(parents=True, exist_ok=True)
    source_path = DATASET_BASE / row.full_path
    image = Image.open(source_path).convert('RGB')
    output = run_sam(image)
    scores = to_numpy(output['scores']).reshape(-1)

    metadata = {
        'selection_id': row.selection_id, 'video_id': row.video_id,
        'timestamp_sec': float(row.timestamp_sec), 'behavior': row.label,
        'lighting': row.lighting, 'source_full_path': row.full_path,
        'annotation_kind': row.annotation_kind, 'prompt': PROMPT,
        'threshold': CONFIDENCE_THRESHOLD, 'num_masks': int(len(scores)),
        'mask_status': 'needs_review',
    }
    image.save(item_dir / 'image.jpg', quality=95)

    if len(scores):
        best = int(scores.argmax())
        mask = np.squeeze(to_numpy(output['masks'])[best]).astype(bool)
        box = to_numpy(output['boxes'])[best].reshape(-1)[:4]
        Image.fromarray(mask.astype(np.uint8)*255).save(item_dir / 'mask.png')
        make_overlay(image, mask).save(item_dir / 'overlay.jpg', quality=95)
        metadata.update({
            'proposal_result': 'mask_proposed', 'selected_index': best,
            'score': float(scores[best]), 'all_scores': [float(x) for x in scores],
            'box_xyxy': [float(x) for x in box], 'mask_pixels': int(mask.sum()),
        })
    else:
        image.save(item_dir / 'overlay.jpg', quality=95)
        metadata.update({'proposal_result': 'no_sam_detection', 'score': None, 'all_scores': []})

    (item_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

gc.collect(); torch.cuda.empty_cache()
print('Batch complete. Re-run the two batch cells to process the next pending items.')

## 6. Consolidate review metadata and create contact sheets

In [ ]:
records = []
for path in PROPOSAL_DIR.glob('*/metadata.json'):
    records.append(json.loads(path.read_text(encoding='utf-8')))
review = pd.DataFrame(records)
if review.empty:
    raise RuntimeError('No proposals exist yet. Run the batch cells first.')
review = review.sort_values(['behavior', 'lighting', 'video_id', 'timestamp_sec'])
REVIEW_CSV = OUTPUT_ROOT / 'review_status.csv'
if REVIEW_CSV.exists():
    old = pd.read_csv(REVIEW_CSV)
    edited = old.set_index('selection_id')[['mask_status']].to_dict()['mask_status']
    review['mask_status'] = review.apply(lambda r: edited.get(r.selection_id, r.mask_status), axis=1)
review.to_csv(REVIEW_CSV, index=False)
print('Completed proposals:', len(review), '/', len(selection))
display(pd.crosstab([review.behavior, review.lighting], review.proposal_result, margins=True))
print('Review CSV:', REVIEW_CSV)

In [ ]:
CONTACT_DIR = OUTPUT_ROOT / 'contact_sheets'
CONTACT_DIR.mkdir(parents=True, exist_ok=True)
cols, rows, thumb, label_h = 5, 5, 220, 42
per_page = cols*rows
for page, start in enumerate(range(0, len(review), per_page), 1):
    chunk = review.iloc[start:start+per_page]
    sheet = Image.new('RGB', (cols*thumb, rows*(thumb+label_h)), 'white')
    draw = ImageDraw.Draw(sheet)
    for i, row in enumerate(chunk.itertuples()):
        overlay_path = PROPOSAL_DIR / row.selection_id / 'overlay.jpg'
        image = Image.open(overlay_path).convert('RGB').resize((thumb, thumb))
        x, y = (i%cols)*thumb, (i//cols)*(thumb+label_h)
        sheet.paste(image, (x, y))
        score = 'none' if pd.isna(row.score) else f'{row.score:.2f}'
        draw.text((x+3, y+thumb+2), f'{row.selection_id} {row.behavior} {row.lighting} score={score}', fill='black')
    sheet.save(CONTACT_DIR / f'review_{page:03d}.jpg', quality=90)
print('Contact sheets:', CONTACT_DIR)
display(Image.open(sorted(CONTACT_DIR.glob('review_*.jpg'))[0]))

## 7. Human review rules and hand-off

Review each overlay and set `mask_status` in `review_status.csv` to one of:

- `accepted`: complete visible civet boundary; no tray, plant, floor or shadow included.
- `corrected`: mask was manually fixed in CVAT.
- `negative`: a person confirmed that no civet is visible.
- `needs_review`: not decided or requires CVAT correction.
- `rejected`: wrong object or unusable image.

Only `accepted`, `corrected` and human-confirmed `negative` items may enter training. Behavior labels also need human review: SAM 3 segments an animal but does not decide Eating, Locomotion or Resting.

After this batch is reviewed, the next step is to package images and proposed masks for CVAT/COCO Segmentation, correct boundary errors, and then train a small segmentation model using video-level train/validation/test splits from the manifest.